# cyfouri-data-explorer: quickstart

The engine of Data Explorer in a notebook. This walkthrough uses the two sample files of the repository (`samples/`): a customer table full of typing traps and a daily sales series.

In [ ]:
from pathlib import Path

import cyfouri_data_explorer as dx

SAMPLES = Path("../../../samples")
ds = dx.open(SAMPLES / "typing_traps.csv")
ds

## Types

The file uses French conventions: decimal commas (`79,50`), `oui`/`non`, dates as `21/02/2019`, percentages. Every column gets a type, a role and the reasons for both.

In [ ]:
dx.to_pandas(ds.types)[["name", "type", "role", "confidence"]]

In [ ]:
ds.column("price_eur").reasons

## Overview and one variable

In [ ]:
overview = ds.overview()
overview.rows, overview.missing_pct, [alert.message for alert in overview.alerts][:5]

In [ ]:
age = ds.variable("age").numeric
normality = [(test.key, test.p_value) for test in age.normality.tests]
age.describe.mean, age.describe.skewness, normality

In [ ]:
dx.to_pandas(ds.fit("price_eur")).head()

## Two variables

The test follows the types: here a number against a category.

In [ ]:
comparison = ds.compare("income", "segment")
dx.to_pandas(comparison)

## Correlations and insights

In [ ]:
correlations = ds.correlations()
[insight.text for insight in correlations.insights][:5]

## Filters

`where()` gives a filtered view; `ds` itself is unchanged.

In [ ]:
france = ds.where("country", "in", ["FR"])
len(ds), len(france), france.variable("income").numeric.describe.median

## Machine learning

In [ ]:
ml = ds.ml("segment")
model = ml.train()
model

In [ ]:
dx.to_pandas(model.explain()).head()

## Synthetic data

In [ ]:
made = ds.synthetic().generate(rows=500)
made.report.scores

## Time series

In [ ]:
sales = dx.open(SAMPLES / "sales_timeseries.csv")
series = sales.timeseries(time="date", value="revenue")
series.setup()

In [ ]:
stationarity = series.stationarity()
stationarity.suggestion

## The HTML report

The same file as the app's **Export report**, without a server.

In [ ]:
ds.report("typing_traps-report.html")[:80]